In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from keras import initializers

Datasets = []
PREDICTORS = ["PwmD", "PwmE"]   
PHYSICAL_PREDICTORS = ["Wd", "We"]   
TARGET_INT = ["Theta", "X", "Y"]  
TARGET = ["DeltaTheta", "DeltaX", "DeltaY"]
     
TIME_STEPS = 3
TS = 0.07


In [2]:
for i in range(4):
    Dataset = pd.read_excel(f"../../../RotedData/Data.xlsx", f"D{i+1}")   
    Datasets.append(Dataset)

for i in range(2):   
    Dataset = pd.read_csv(f"../../../Data/Data{i + 1}.csv")  
    Datasets.append(Dataset)
    
    
for i in range(len(Datasets)):
    Dataset = Datasets[i].copy()

    for var in TARGET_INT:
        Dataset[f"Delta{var}"] = (Dataset[var].shift(-1) - Dataset[var]) / TS
    
    for var in PREDICTORS:
        Dataset[f"s{var}"] = (Dataset[var].shift(-1) + Dataset[var]) 
    
    for var in PREDICTORS:
        Dataset[f"d{var}"] = (Dataset[var].shift(-1) - Dataset[var])  
            
    Dataset = Dataset.dropna(subset=[f"Delta{var}" for var in TARGET_INT])

    Datasets[i] = Dataset

In [5]:
PREDICTORS = ["dPwmD", "dPwmE", "sPwmD", "sPwmE"]   
INPUT_SIZE = len(PREDICTORS)  
OUTPUT_SIZE = len(TARGET)   

In [6]:
NormDatasets = []

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

Train1 = Datasets[0].copy()
Train1[PREDICTORS] = SCALER.fit_transform(Train1[PREDICTORS])
Train1[TARGET] = OUT_SCALER.fit_transform(Train1[TARGET])
NormDatasets.append(Train1)

Train2 = Datasets[1].copy()
Train2[PREDICTORS] = SCALER.transform(Train2[PREDICTORS])
Train2[TARGET] = OUT_SCALER.transform(Train2[TARGET])
NormDatasets.append(Train2)

# concatena
Train = pd.concat([Train2, Train1], ignore_index=True)

for i in range(4):
    CurrentTestDataset = Datasets[i + 2].copy()
    CurrentTestDataset[PREDICTORS] = SCALER.transform(CurrentTestDataset[PREDICTORS])
    CurrentTestDataset[TARGET] = OUT_SCALER.transform(CurrentTestDataset[TARGET])
    NormDatasets.append(CurrentTestDataset)

In [7]:
def CreateSequences(input_data, target_data, timesteps):
    X_seq, Y_seq = [], []
    
    for i in range(timesteps, len(input_data)):
        X_seq.append(input_data.iloc[i-timesteps:i].values)
        Y_seq.append(target_data.iloc[i])
    return np.array(X_seq), np.array(Y_seq)

In [8]:
x_train, y_train = CreateSequences(Train[PREDICTORS], Train[TARGET], TIME_STEPS)

x_val, y_val = CreateSequences((NormDatasets[5])[PREDICTORS], (NormDatasets[5])[TARGET], TIME_STEPS)
print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

print(f"Dimensão da entrada: {np.shape(x_val)}")
print(f"Dimensão da saida: {np.shape(x_val)}")

Dimensão da entrada: (1949, 3, 4)
Dimensão da saida: (1949, 3)
Dimensão da entrada: (1268, 3, 4)
Dimensão da saida: (1268, 3, 4)


In [9]:
Wd_train =  Train["Wd"].values[:len(x_train)]
We_train =  Train["We"].values[:len(x_train)]
theta_train = Train["Theta"].values[:len(x_train)]

In [10]:
print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

print(f"Dimensão da entrada fisica : {np.shape(Wd_train)}")
print(f"Dimensão da entrada fisica: {np.shape(We_train)}")

Dimensão da entrada: (1949, 3, 4)
Dimensão da saida: (1949, 3)
Dimensão da entrada fisica : (1949,)
Dimensão da entrada fisica: (1949,)


In [11]:
import matplotlib.pyplot as plt
import numpy as np

def PlotOut(ax, title, target_name, y_true, y_cin, y_pred):

    time = (np.arange(0, len(y_pred), 1).astype(float) * 0.07).round(5)

    ax.scatter(time, y_true,
               marker='o',
               s=12,
               color='tab:blue',
               label='Amostras Reais',
               alpha=0.7)

    ax.scatter(time, y_pred,
               marker='x',
               s=12,
               color='tab:red',
               label='Modelo PIRNN',
               alpha=0.7)

    ax.plot(time, y_cin,
            linewidth=2,
            color='tab:green',
            label='Modelo Cinemático')

    ax.set_title(f'{title} - {target_name}')
    ax.set_xlabel('Tempo [s]')
    ax.set_ylabel(target_name)

    ax.legend()
    ax.grid(True)

In [12]:
import tensorflow as tf

R = tf.constant(0.0341, dtype=tf.float32)
L = tf.constant(0.0606, dtype=tf.float32)
dt = tf.constant(TS, dtype=tf.float32)

def NumericalIntegration(dataset, dq):

    q = [None] * OUTPUT_SIZE

    init_vals = np.array([
        dataset[name].iloc[0] for name in TARGET_INT
    ])

    for j in range(OUTPUT_SIZE):
        q[j] = init_vals[j] + np.cumsum(dq[j] * TS)

    return q


In [13]:
def CinematicModel(Wd, We, theta):

    dtheta_cin = (R / (2 * L)) * (Wd - We)
    dx_cin = (R / 2) * tf.cos(theta) * (Wd + We)
    dy_cin = (R / 2) * tf.sin(theta) * (Wd + We)

    return [dtheta_cin, dx_cin, dy_cin]

In [14]:
class PINN(tf.keras.Model):

    def __init__(self, architecture, initializer):
        super(PINN, self).__init__()

        self.rnn_layers = []

        for i, units in enumerate(architecture):

            if i < len(architecture) - 1:
                self.rnn_layers.append(
                    tf.keras.layers.SimpleRNN(
                        units,
                        activation='tanh',
                        return_sequences=True,
                        kernel_initializer=initializer
                    )
                )
            else:
                self.rnn_layers.append(
                    tf.keras.layers.SimpleRNN(
                        units,
                        activation='tanh',
                        kernel_initializer=initializer

                    )
                )

        # camada densa final
        self.out_layer = tf.keras.layers.Dense(OUTPUT_SIZE,
                                               activation="linear",                 kernel_initializer = initializer
)

    def call(self, inputs):

        x = inputs

        for rnn in self.rnn_layers:
            x = rnn(x)

        return self.out_layer(x)

In [15]:
mean = OUT_SCALER.mean_[0]
std  = OUT_SCALER.scale_[0]
mean_tf = tf.constant(mean, dtype=tf.float32)
std_tf  = tf.constant(std, dtype=tf.float32)

In [16]:
@tf.function
def train_step(model, optimizer, x, y, Wd, We, theta):

    with tf.GradientTape() as tape:

        pred = model(x, training=True)

        # loss dos dados
        data_loss = tf.reduce_mean(tf.square(pred - y))
        
        # termo físico
        physics = tf.stack(CinematicModel(Wd, We, theta), axis=1)   
             
        # normalização correta
        physics_denorm = (physics - mean_tf) / std_tf

        physics_loss = tf.reduce_mean(tf.square(pred - physics_denorm))

        loss = data_loss + physics_loss

    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    return loss

In [17]:
Wd_train = tf.convert_to_tensor(Wd_train, dtype=tf.float32)
We_train = tf.convert_to_tensor(We_train, dtype=tf.float32)
theta_train = tf.convert_to_tensor(theta_train, dtype=tf.float32)

x_train_tf = tf.convert_to_tensor(x_train, dtype=tf.float32)
y_train_tf = tf.convert_to_tensor(y_train, dtype=tf.float32)

x_val_tf = tf.convert_to_tensor(x_val, dtype=tf.float32)
y_val_tf = tf.convert_to_tensor(y_val, dtype=tf.float32)



In [18]:
def EarlyStopping(model, best_loss, counter, best_weights, min_delta=1e-3):
    val_pred = model(x_val_tf)
    val_loss = tf.reduce_mean(tf.square(val_pred - y_val_tf))
    
    if val_loss < (best_loss - min_delta):
        best_loss = val_loss
        counter = 0
        best_weights = model.get_weights()

    else:
        counter += 1

    return best_loss, counter, best_weights, val_loss

def TrainPINN(model, optimizer, patience=500, best_loss=np.inf):
    counter = 0
    best_weights = model.get_weights()

    for epoch in range(20000):

        loss = train_step(model, optimizer, x_train_tf, y_train_tf, Wd_train, We_train, theta_train)
        best_loss, counter, best_weights, val_loss =  EarlyStopping(model, best_loss, counter, best_weights)
        
        if counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            model.set_weights(best_weights)
            break

        if epoch % 100 == 0:
            print(f"Epoch {epoch} | Train Loss {loss.numpy():.6f} | Val Loss {val_loss.numpy():.6f}")

In [19]:
def GetCin(dataset): 
    dq = CinematicModel(tf.convert_to_tensor(dataset["Wd"].values, dtype=tf.float32),
                        tf.convert_to_tensor(dataset["We"].values, dtype=tf.float32), 
                        tf.convert_to_tensor(dataset["Theta"].values, dtype=tf.float32))
    q = NumericalIntegration(dataset, dq)
    return np.vstack(q).T

In [20]:
TITLES = ["train1", "train2", "test1", "test2", "test3", "val"]

def EvalModel(model):

    n_datasets = len(NormDatasets)
    n_targets = len(TARGET)

    fig, axs = plt.subplots(
        n_datasets,
        n_targets,
        figsize=(6*n_targets, 4*n_datasets)
    )

    # estrutura das métricas
    metrics = {
        name: {
            "R2_train1": [], "R2_train2": [],
            "R2_val": [],
            "R2_test1": [], "R2_test2": [], "R2_test3": [],
            "MSE_train1": [], "MSE_train2": [],
            "MSE_val": [],
            "MSE_test1": [], "MSE_test2": [], "MSE_test3": [],
        }
        for name in TARGET_INT
    }

    for i, dataset in enumerate(NormDatasets):

        x = dataset[PREDICTORS]
        y = dataset[TARGET]

        x, y = CreateSequences(x, y, TIME_STEPS)

        # predição da rede
        pred = model(tf.convert_to_tensor(x, dtype=tf.float32)).numpy()

        # desnormalização
        y_pred_diff = OUT_SCALER.inverse_transform(pred)
        y_diff = OUT_SCALER.inverse_transform(y)

        # arrays reconstruídos
        y_true = np.zeros_like(y_diff)
        y_pred = np.zeros_like(y_pred_diff)
        y_cin = GetCin(Datasets[i])
        
        n = y_true.shape[0]
        y_cin = y_cin[:n]

        # valores iniciais reais
        init_vals = np.array([
            Datasets[i][name].iloc[0] for name in TARGET_INT
        ])

        # reconstrução por integração cumulativa
        for j in range(n_targets):
            y_true[:, j] = init_vals[j] + np.cumsum(y_diff[:, j] * TS)
            y_pred[:, j] = init_vals[j] + np.cumsum(y_pred_diff[:, j] * TS)

        # cálculo das métricas
        for j, name in enumerate(TARGET_INT):

            r2 = r2_score(y_true[:, j], y_pred[:, j])
            mse = mean_squared_error(y_true[:, j], y_pred[:, j])

            metrics[name][f"R2_{TITLES[i]}"].append(r2)
            metrics[name][f"MSE_{TITLES[i]}"].append(mse)

            print(
                f"{name} | {TITLES[i]} -> "
                f"R² = {r2:.4f}, MSE = {mse:.4e}"
            )

            # seleção robusta do eixo
            if n_datasets > 1 and n_targets > 1:
                ax = axs[i, j]
            elif n_datasets > 1:
                ax = axs[i]
            elif n_targets > 1:
                ax = axs[j]
            else:
                ax = axs

            PlotOut(
                ax,
                TITLES[i],
                name,
                y_true[:, j],
                y_cin[:, j],
                y_pred[:, j],
                
            )

    plt.tight_layout()
    return metrics, plt, fig

In [21]:
from tensorflow.keras.optimizers import Adam

N_MODELS = 10  # número de inicializações
seeds = np.random.choice(range(1, 10000), size=N_MODELS, replace=False)

architectures = [[64], [128], [264],
                 [8, 4], [16, 8], [32, 16], [64, 32], [128, 64], [264, 128],
                 [16, 8, 4], [32, 16, 8], [128, 64, 32], [264, 128, 64] ] 

results = {}
excel_file = "resultados.xlsx"

for arch in architectures:
    for i, s in enumerate(seeds):

        tf.keras.backend.clear_session()

        init = initializers.RandomNormal(seed=int(s))

        model = PINN(architecture=arch, initializer=init)

        model.build((None, TIME_STEPS, len(PREDICTORS)))

        w0 = model.get_weights()

        opt = Adam(learning_rate=0.001)
        opt.build(model.trainable_variables)
        
        TrainPINN(model, optimizer=opt)
        
        wf = model.get_weights()

        metrics, plt, fig = EvalModel(model)
        
        arch_name = "_".join(map(str, arch))
        row = {
            "model": f"model_{arch_name}_{i}",
            "Neurons": arch,
            "W0": str([w.round(4).tolist() for w in w0]),
            "Wf": str([w.round(4).tolist() for w in wf]),
        }


        for name in TARGET_INT:
            row.update({
                f"R2_Train_1_{name}": metrics[name]["R2_train1"],
                f"MSE_Train_1_{name}": metrics[name]["MSE_train1"],
                f"R2_Train_2_{name}": metrics[name]["R2_train2"],
                f"MSE_Train_2_{name}": metrics[name]["MSE_train2"],
                f"R2_Val_{name}": metrics[name]["R2_val"],
                f"MSE_Val_{name}": metrics[name]["MSE_val"],
                f"R2_Test_1_{name}": metrics[name]["R2_test1"],
                f"MSE_Test_1_{name}": metrics[name]["MSE_test1"],
                f"R2_Test_2_{name}": metrics[name]["R2_test2"],
                f"MSE_Test_2_{name}": metrics[name]["MSE_test2"],
                f"R2_Test_3_{name}": metrics[name]["R2_test3"],
                f"MSE_Test_3_{name}": metrics[name]["MSE_test3"],
            })
        
        save_fig = any(
            any(v > 0.8 for v in metrics[name][key])
            if isinstance(metrics[name][key], (list, tuple))
            else metrics[name][key] > 0.5
            for name in TARGET_INT
            for key in metrics[name]
            if key.startswith("R2_test"))
    

        if save_fig:
            plt.savefig(
                f"./results/model_{arch_name}_{i}.pdf",
                format="pdf",
                bbox_inches="tight"
            )
            print(f"Figura salva em: ./results/model_{arch_name}_{i}.pdf")

        plt.close(fig)
        # plt.show()

        df = pd.DataFrame([row])

        # salva/atualiza Excel incrementalmente
        try:
            # tenta abrir existente e adicionar linha
            old = pd.read_excel(excel_file)
            new_df = pd.concat([old, df], ignore_index=True)
            new_df.to_excel(excel_file, index=False)
        except FileNotFoundError:
            # se não existir, cria arquivo novo
            df.to_excel(excel_file, index=False)

        print(f"Modelo {i} treinado e salvo no Excel")



Epoch 0 | Train Loss 1.581855 | Val Loss 0.782289
Epoch 100 | Train Loss 1.238595 | Val Loss 0.697930
Epoch 200 | Train Loss 1.220530 | Val Loss 0.697379
Epoch 300 | Train Loss 1.196636 | Val Loss 0.684730
Epoch 400 | Train Loss 1.148232 | Val Loss 0.641922
Epoch 500 | Train Loss 1.113392 | Val Loss 0.631039
Epoch 600 | Train Loss 1.081683 | Val Loss 0.645343
Epoch 700 | Train Loss 1.054196 | Val Loss 0.649421
Epoch 800 | Train Loss 1.026004 | Val Loss 0.646002
Epoch 900 | Train Loss 0.996728 | Val Loss 0.646448
Early stopping at epoch 959
Theta | train1 -> R² = 0.9536, MSE = 1.0581e-02
X | train1 -> R² = -0.2042, MSE = 1.2984e-01
Y | train1 -> R² = -0.6099, MSE = 1.3490e-01
Theta | train2 -> R² = 0.7287, MSE = 6.1928e-02
X | train2 -> R² = 0.0202, MSE = 8.2557e-02
Y | train2 -> R² = -0.9935, MSE = 2.1465e-01
Theta | test1 -> R² = -0.4090, MSE = 3.2368e-01
X | test1 -> R² = -0.0089, MSE = 1.0569e-01
Y | test1 -> R² = -1.6333, MSE = 2.1579e-01
Theta | test2 -> R² = 0.8142, MSE = 4.405

C:\Users\Sarah\AppData\Local\Temp\ipykernel_9144\1563548962.py:93: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
C:\Users\Sarah\AppData\Local\Temp\ipykernel_9144\1772612896.py:70: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(


Figura salva em: ./results/model_32_16_8_9.pdf
Modelo 9 treinado e salvo no Excel
Epoch 0 | Train Loss 1.563964 | Val Loss 0.794548
Epoch 100 | Train Loss 1.226841 | Val Loss 0.664916
Epoch 200 | Train Loss 1.145870 | Val Loss 0.635750
Epoch 300 | Train Loss 1.074326 | Val Loss 0.631567
Epoch 400 | Train Loss 1.017327 | Val Loss 0.656630
Epoch 500 | Train Loss 0.960019 | Val Loss 0.681270
Epoch 600 | Train Loss 0.915537 | Val Loss 0.723691
Epoch 700 | Train Loss 0.875255 | Val Loss 0.727637
Early stopping at epoch 743
Theta | train1 -> R² = 0.9711, MSE = 6.5918e-03
X | train1 -> R² = -0.0915, MSE = 1.1769e-01
Y | train1 -> R² = -0.6219, MSE = 1.3591e-01
Theta | train2 -> R² = 0.7399, MSE = 5.9374e-02
X | train2 -> R² = -0.1766, MSE = 9.9140e-02
Y | train2 -> R² = -1.0300, MSE = 2.1858e-01
Theta | test1 -> R² = 0.0047, MSE = 2.2865e-01
X | test1 -> R² = -0.0424, MSE = 1.0921e-01
Y | test1 -> R² = -1.7593, MSE = 2.2612e-01
Theta | test2 -> R² = 0.8601, MSE = 3.3181e-02
X | test2 -> R² = 